# nib — Colab evaluation

**This notebook contains no logic.** It clones, installs, mounts Drive, copies
one file to local disk, and calls scripts. Every decision lives in
`configs/base.yaml` and every line of code lives in the repository.

## What this run is for

On 2026-09-11 the project's style metric was found to be measuring the wrong
thing. A **real** line, by unquestionably the right writer, blurred by 0.8
pixels — damage that does not change whose handwriting it is — scored 12.2%
where the untouched line scored 96.8%. Every generative decoder produces
exactly that softness, so writer retrieval has been reporting sharpness as much
as style, and every style comparison this project has made is confounded by it.

`HWD` replaces it. It barely moves under the same damage, it separates
handwriting from a typeface by a wide margin, and it is what Emuru's and Eruku's
own papers report.

**Its scale is measured inside the run, not quoted.** HWD compares each writer's
*average* look, and an average over few lines is noisy, so what real handwriting
scores depends on how many lines each writer contributes. Measured on our data:

```
lines per writer     1      2      3      6      10
real vs real       1.76   1.26   1.01   0.70   0.54
```

The 0.641 quoted earlier sits between six and ten lines per writer. A 300-sample
run gives about three, where real handwriting itself scores about 1.06. So every
run now reports three figures against one shared set of real lines:

- **generated** — what the model wrote
- **real** — the target lines themselves: same writers, same texts, same counts
- **typeface** — the same texts in a font: no hand at all

and, for each, whether it is closer to its own writer than to everyone else —
the **identity** block, which tells "not this writer" apart from "not real
handwriting".

## Run cells 1 to 6 in order

Cell 6 saves its own run to Drive. Cell 7b (two style lines), 7c (quality
control), 7d (per-writer fine-tune), 7e (keep the closest hand), 7f (fine-tune, strokes withheld), 7g (Amri's own hand) and cell 8 (Eruku) are optional and save themselves too; cell 7 copies everything again if a run
did not. Anything below cell 9 is kept for reference and should not be run
without a reason.

## Before you start

Under `MyDrive/nib/` you need `cvl_lines_64.lmdb` (**127 MB, 9,142 records** —
the copy from `data/processed/upload/`, never the 8 GB one) and
`checkpoints/writer_embedder.pt`.

## 1. Clone and install

`torch` is deliberately absent — Colab's build is matched to its CUDA driver.

The `hwd` extra is new and it is a large install: the package imports every
score it owns at import time, so it pulls in gudhi, matplotlib, tiktoken and
scikit-learn. Two or three minutes.

**Expect a pip conflict warning about `gradio`.** Installing `transformers<5`
pulls `huggingface-hub` down to the range it needs and Colab's preinstalled
gradio wants something newer. Nothing here imports gradio. Cell 2 is the check
that decides.

In [ ]:
REPO_URL = "https://github.com/omritzabari/nib.git"

%cd /content
![ -d nib ] || git clone $REPO_URL nib
%cd /content/nib
!git pull --ff-only
!pip install -q -e ".[dev,track,models,hwd]"

## 2. What are we running on

**Stop here if any of these is wrong.** A rebuilt Colab VM arrives without an
accelerator unless one is asked for, and generation on CPU is 220 seconds a
line — 18 hours for 300.

- `Tesla T4` must appear
- `torch` must say `+cu128`, not `+cpu`
- `hwd available: True` and `hwd imports cleanly` — otherwise HWD is skipped,
  and the second of those two lines prints the reason instead

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
import transformers

print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, " <- must be 4.x")
assert transformers.__version__.startswith("4."), "5.x cannot load Emuru; see pyproject"

# In a fresh interpreter, not in this kernel: the kernel started before the
# install and has /content on its path, where the clone reads as an empty package
# named `nib`. The scripts always run in a fresh interpreter, so this checks them.
!cd /content/nib && python -c "from nib.engine.metrics import hwd; print('hwd available:', hwd.available(), ' <- must be True')"
!cd /content/nib && python -c "import hwd.scores" && echo "hwd imports cleanly"

## 3. Mount Drive and copy what the run needs

One sequential copy of one file. Reading the pack record-by-record over Drive
would leave the GPU waiting on network round-trips.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/nib/data/processed /content/nib/checkpoints
!time cp /content/drive/MyDrive/nib/cvl_lines_64.lmdb /content/nib/data/processed/
!cp /content/drive/MyDrive/nib/checkpoints/writer_embedder.pt /content/nib/checkpoints/
!ls -lh /content/nib/data/processed/ /content/nib/checkpoints/

## 4. Is everything here

Two things will read as missing and both are correct: the **word pack**,
because one pack is required and this session needs lines; and the **raw CVL
images**, because 5 GB of sources are not copied to a VM that only reads a
127 MB pack. The only consequence is that CER cannot be re-measured here, and
it has already been measured where the sources are.

In [ ]:
%cd /content/nib
!python scripts/check_data.py

## 5. Check the harness

The `fake` generator draws the target text in a typeface. Every number it
produces is meaningless and every shape is right, which is what a pipeline
check needs. It has caught an unexercised code path twice.

The first run on a fresh VM downloads HWD's VGG weights, 675 MB.

**What must appear**, or something below is not wired:

- square brackets after every figure — the 95% spreads
- an `HWD` block with `generated`, `real` and `typeface` lines — not
  `not measured`, not `FAILED`
- `generated` equal to `typeface`, and `100%` of the way — for this generator
  they are the very same images, so anything else is a bug
- in the `identity` block, `generated` and `typeface` identical again with a gap
  near zero, and `real` clearly above both
- a line beginning `generated` with a folder, and one beginning `analysis`

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 120 --device cuda

## 6. Emuru with a working style metric — the main run

Generation took 53.9 minutes for 300 lines on the last run, then the metrics.

Two blocks under `HWD` matter:

- **distance** — `generated` against `real` and `typeface` from the same run.
  Emuru read 2.00 against 0.86 and 2.99: 53% of the way to no hand at all.
- **identity** — whether each set is closer to *its own* writer than to the
  others. Distance alone cannot tell "not this writer" from "not real
  handwriting"; this can. The last line gives the share of the real lines'
  identity the generated lines carry: 0% is nobody's hand in particular, 100% is
  as distinct as the writer's own lines. `typeface` should sit near a gap of zero.

`truncated` sat at 4.4% last time.

**The cell saves to Drive by itself when the run ends** and prints how many
generated images arrived — about 295.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator emuru \
    --samples 300 \
    --device cuda

# To Drive the moment the run ends. The VM that produced the first trustworthy
# style figure was reclaimed before anyone copied it.
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines/generated | wc -l) generated images"

## 7. Save everything again — only if a run above did not

Cells 6, 7b and 8 save their own run to Drive when they finish. This copies
every run and the references, and is safe to repeat.

In [ ]:
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_* /content/drive/MyDrive/nib/results/
!cp -r /content/nib/references /content/drive/MyDrive/nib/results/
!du -sh /content/drive/MyDrive/nib/results/*
!for f in /content/nib/outputs/eval_*/results.json; do echo "== $f"; cat "$f"; done

## 7b. Emuru with two style lines — optional, about as long as cell 6

The model is given one line of a hand and asked to learn it; a real user brings
a page. Two lines generated normally on 2026-09-10 (0.85x of real width) while
four broke Emuru's stopping rule, so two is the most evidence it can take as it
stands.

Compare its `identity` line with cell 6's. If the intervals separate, more
evidence of the hand helps, and fixing the stopping rule to allow more lines is
worth the work. If they overlap, it does not, at least at two.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator emuru \
    --samples 300 \
    --style-refs 2 \
    --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs2 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs2/generated | wc -l) generated images"

## 7c. Quality control — several draws per line, the broken ones rejected

Emuru's output is bimodal: the typical line reads well, and about one in ten
collapses into a smear or comes out blank. This draws up to four candidates per
line, each from **one** of four style lines by that writer, reads each with
TrOCR-small, and keeps the first readable one. TrOCR-base still measures CER, so
the selector is not also the judge.

The first command is a check on the fake generator and takes a few minutes, most
of it downloading TrOCR-small and TrOCR-base. The Emuru run starts only if the
check succeeds. How long the Emuru run takes depends on how many lines need a
second draw; the `selection` line reports it.

**Compare with cell 6** — identity 56.5% [50.9, 61.9], CER 30.4%, FID 67.70:

- `HWD identity` and `FID` are the clean measures; no recogniser touches them
- `selection` says how many draws were spent
- `CER` should fall, but it is partly flattered by selection

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 60 --style-refs 4 --candidates 2 --device cuda \
  && python scripts/evaluate_generator.py --generator emuru --samples 300 --style-refs 4 --candidates 4 --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs4_cand4 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs4_cand4/generated | wc -l) generated images"

## 7d. Per-writer fine-tune — same writers, same lines, with and without

For 24 held-out writers: 16 of their lines are the "page", 4 are generated, the
rest are the reference. The 4 are generated with Emuru as released, then again
after a short LoRA fine-tune on the 16, both with the quality control of 7c.
The adapter is reset before the next writer.

After the first writer a progress line prints the minutes to go; the training
time on a T4 has not been measured yet, and it is one of the things this cell
is for.

**What to read:**

- `difference` under `HWD identity` — fine-tuned minus released, paired by
  writer. If its interval is above zero, the fine-tune makes the output more
  like the writer
- `training` — seconds per writer, the cost per user
- `CER` for both — the fine-tune must not make the text unreadable

In [ ]:
%cd /content/nib
!python scripts/evaluate_finetune.py --writers 24 --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/finetune_w24_t16_s150_r8 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/finetune_w24_t16_s150_r8/adapted | wc -l) adapted images"

## 7e. Keep the draw closest to the hand — 150 lines

7c kept the first readable draw. This makes all four draws for every line, sets
the unreadable ones aside, and keeps the one whose writer embedding is closest to
the writer's style lines. The embedding chooses; HWD, a different network,
measures — so `writer top-1` in this run is not independent and should not be
read.

150 lines rather than 300, because every line now costs four draws. They are the
same first 150 requests 7c made, so the two runs can be compared line for line
afterwards, without a GPU.

The first command is the fake-generator check; Emuru starts only if it passes.

**Send back:** the `HWD` block with `identity`, the `SUMMARY`, the `selection`
lines and `saved to Drive`.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 60 --style-refs 4 --candidates 2 --keep hand --device cuda \
  && python scripts/evaluate_generator.py --generator emuru --samples 150 --style-refs 4 --candidates 4 --keep hand --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs4_cand4_byhand /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs4_cand4_byhand/generated | wc -l) generated images"

## 7f. Per-writer fine-tune, with the writer's own strokes withheld

7d trained on the writer's lines alone and changed nothing: identity +0.1 points
[-6.8, 6.7]. The likely reason is that every stroke was predicted with the
writer's own earlier strokes in view, so nothing had to be stored in the
adapter. Here each of the writer's lines is placed after a line by another
writer, the loss counts only the writer's line, and the teacher-forced slices
get noise 0.5 instead of 0.1 (latent ink measures a standard deviation of about
1.17). Everything else is 7d: the same 24 writers, lines and targets.

Should take about as long as 7d, which was 71 minutes; a pair is a wider image,
so training may be slower. The progress line after the first writer will say.

**Send back:** the first writer's line, then the `HWD identity` and `CER` blocks,
the `SUMMARY` and `saved to Drive`.

In [ ]:
%cd /content/nib
!python scripts/evaluate_finetune.py --writers 24 --context other --noise 0.5 --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/finetune_w24_t16_s150_r8_other_n0.5 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/finetune_w24_t16_s150_r8_other_n0.5/adapted | wc -l) adapted images"

## 7g. Amri's own hand — one photographed page

The whole system on a real user for the first time. Needs
`MyDrive/nib/personal/passage_page1.jpg` — the photo of page 1 of the dictated
passage.

The page is split into its 22 lines. Lines 2, 6 and 18 carry corrections and
19-22 have small marks the segmentation assigns to the wrong line, so those are
set aside. Of the 15 left, 10 are the page the system learns the hand from and
5 are written again without being shown, for comparison. Then all 22 lines of
page 2 — a text Amri never wrote — are written in his hand.

Every line keeps the draw closest to the hand (`--keep hand`), which raised
identity to 69.8% in 7e. About 110 draws in all; at the 10.6 s a draw measured
in 7e, roughly 20 minutes.

**Look at, and send back:** `comparison.png` and `written.png` (they are copied
to Drive), the `CER on the targets` line and the `selection` lines. The `blind/`
folder is for showing to people who know the handwriting.

In [ ]:
%cd /content/nib
!mkdir -p /content/nib/data/raw/personal
!cp /content/drive/MyDrive/nib/personal/passage_page1.jpg /content/nib/data/raw/personal/
!python scripts/probe_writer.py --photo data/raw/personal/passage_page1.jpg --skip 2,6,18,19,20,21,22 --keep hand --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/probe_passage_page1 /content/drive/MyDrive/nib/results/
!ls /content/drive/MyDrive/nib/results/probe_passage_page1

## 7h. DiffBrush — set up, once per session

DiffBrush (ICCV 2025) is a diffusion model: it samples each line instead of
averaging, which is what Emuru's vanishing dots and colons point at. Its code is
cloned at the commit the adapter was written against, and its 1.17 GB checkpoint
is kept on Drive after the first download.

Run cells 1-4 first. The first time, this downloads the checkpoint from the
authors' Google Drive into `MyDrive/nib/checkpoints/`; after that it only copies.
If the download fails (Google Drive sometimes refuses large files), download
`DiffBrush-ckpt.pt` by hand from
https://drive.google.com/file/d/1EWzBmLtnQ42cTf3k_CYQ-nF3RXCb35I6 and put it in
`MyDrive/nib/checkpoints/`.

**Must appear:** `DiffBrush at da9addc` and a checkpoint of `1165546469` bytes.

In [ ]:
%cd /content/nib
!mkdir -p third_party checkpoints/diffbrush /content/drive/MyDrive/nib/checkpoints
![ -d third_party/DiffBrush ] || git clone -q https://github.com/dailenson/DiffBrush.git third_party/DiffBrush
!cd third_party/DiffBrush && git checkout -q da9addc && git log -1 --format="DiffBrush at %h, %ad" --date=short
![ -f /content/drive/MyDrive/nib/checkpoints/DiffBrush-ckpt.pt ] || curl -sSL -o /content/drive/MyDrive/nib/checkpoints/DiffBrush-ckpt.pt "https://drive.usercontent.google.com/download?id=1EWzBmLtnQ42cTf3k_CYQ-nF3RXCb35I6&export=download&confirm=t"
!cp /content/drive/MyDrive/nib/checkpoints/DiffBrush-ckpt.pt checkpoints/diffbrush/
!ls -l checkpoints/diffbrush/

## 7i. DiffBrush on 150 CVL lines — stage 2, no fine-tune

The same first 150 requests as 7c and 7e, the same quality control as 7c: four
style lines a request, up to four draws, the first readable one kept. So its
identity can be set beside Emuru's line for line afterwards, without a GPU.

DiffBrush was trained on IAM only, and CVL writers are new to it: **a figure
below Emuru's is expected here** and does not end the experiment. The real test
is stage 3, a short fine-tune per writer. On CPU a line took 38 s; the first
progress line gives the T4 figure.

**Send back:** the first progress line, the `HWD` block with `identity`, the
`SUMMARY`, the `truncated`, `empty outputs` and `selection` lines, and
`saved to Drive`.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator diffbrush --samples 150 --style-refs 4 --candidates 4 --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_diffbrush_lines_refs4_cand4 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_diffbrush_lines_refs4_cand4/generated | wc -l) generated images"

## 7j. DiffBrush on Amri's page

Exactly 7g with DiffBrush writing: the same 10 lines to learn from, the same 5
written again, all 22 lines of page 2, the draw closest to the hand kept.
Written to `results/probe_passage_page1_diffbrush/`, so Emuru's page stays.

**Look at** `comparison.png` and `written.png` beside Emuru's. **Send back** the
`CER on the targets` line and the `selection` lines.

In [ ]:
%cd /content/nib
!mkdir -p /content/nib/data/raw/personal
!cp /content/drive/MyDrive/nib/personal/passage_page1.jpg /content/nib/data/raw/personal/
!python scripts/probe_writer.py --generator diffbrush --photo data/raw/personal/passage_page1.jpg --skip 2,6,18,19,20,21,22 --keep hand --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/probe_passage_page1_diffbrush /content/drive/MyDrive/nib/results/
!ls /content/drive/MyDrive/nib/results/probe_passage_page1_diffbrush

## 7k. DiffBrush fine-tuned per writer — stage 3, the one that decides

Needs 7h. The same 24 writers, 16 train lines and 4 targets as 7d. DiffBrush
writes the targets as released, then after a LoRA fine-tune of 300 steps on the
writer's train lines, then the adapter is reset for the next writer. Both go
through quality control.

Emuru's released images from 7d (`results/finetune_w24_t16_s150_r8` on Drive) are
scored as a third condition against the same reference. **Emuru's identity should
come out close to 7d's 53.7%**; if it does not, the reference differs and the
comparison is void.

**The criterion, set before the run:** `fine-tuned minus Emuru released` with its
whole interval above zero. Anything else closes DiffBrush at these settings.

How long it takes is not known: the first writer's progress line gives the time
a writer, and the rest follows from it.

**Send back:** the first writer's line, the `HWD identity` and `CER` blocks, the
`SUMMARY` and `saved to Drive`.

In [ ]:
%cd /content/nib
!python scripts/evaluate_finetune_diffbrush.py --writers 24 --device cuda --compare-with /content/drive/MyDrive/nib/results/finetune_w24_t16_s150_r8

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/finetune_diffbrush_w24_t16_s300_r8 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/finetune_diffbrush_w24_t16_s300_r8/adapted | wc -l) adapted images"

## 7l. The same fine-tune, gentler — the same 24 writers, about 20 minutes

7k trained too hard: the words came out as the right words drawn as mush, CER
12.1% → 34.6%, and identity fell. Three changes at once, because this is a
decision gate and not an investigation:

- **60 steps at 2e-5**, against 300 at 1e-4 — seven or eight passes over each
  line instead of thirty-seven;
- **the adapter only on `attn1`**, the attention over the drawing itself, so the
  layers that tie it to the glyphs are left alone;
- **training lines wider than the canvas left out** rather than squeezed; 67 of
  384 were squeezed in 7k, and a writer left with fewer than two narrow lines
  keeps the wide ones anyway.

The same 24 writers as 7k and 7d — asking for fewer would draw a different
sample and Emuru would have no images for them. Training drops from 52 s a writer
to about 10, so the run is roughly 20 minutes rather than 36.

**The criterion is unchanged:** `fine-tuned minus Emuru released` wholly above
zero, with Emuru's own figure again near 53.7%. If it is not, DiffBrush closes.

**Send back:** the first writer's line, the `HWD identity` and `CER` blocks and
the `SUMMARY`.


In [ ]:
%cd /content/nib
!python scripts/evaluate_finetune_diffbrush.py --writers 24 --device cuda --compare-with /content/drive/MyDrive/nib/results/finetune_w24_t16_s150_r8

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/finetune_diffbrush_w24_t16_s60_r8 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/finetune_diffbrush_w24_t16_s60_r8/adapted | wc -l) adapted images"


## 7m. Choose the style line by its letters — no training, 150 lines

The model is handed **one** of the writer's lines per draw, so a page of twenty
reaches it one line at a time. Until now that line was chosen by width alone.
This chooses, among lines of a workable width, the one that shows most of the
characters the target needs: if the line to write is full of `y` and `g`, it sees
a line of theirs that has them.

Nothing is trained. The same first 150 requests as 7c and 7e, the same quality
control as 7c, so the only change against 7c is which line is shown.

**Compare with 7c's first 150 lines: identity 64.6%**, measured line for line
against one reference. **Send back** the `HWD identity` block, the `SUMMARY` and
the `selection` lines.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator emuru --samples 150 --style-refs 4 --candidates 4 --style-by letters --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs4_cand4_styleletters /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs4_cand4_styleletters/generated | wc -l) generated images"

## 7n. Teach Emuru real handwriting, once — about 12 minutes

Emuru was pre-trained on millions of lines rendered from fonts and has **never
seen a pen**. Everything it knows about imitating a hand, it learned on printed
imitations of handwriting. This trains one adapter on 4,000 real lines by the
**216 training-split writers** -- never on the 94 the project measures on, so the
figures stay honest.

This is not the per-writer fine-tune. Nothing about a particular writer is meant
to be learned, and nothing is trained at enrolment: the adapter ships with the
system, and a new user still just brings their page.

The adapter is a few megabytes and is copied to Drive, so it is trained once and
loaded for ever.

**Send back:** the `lines`, `LoRA` and `passes` lines, the `loss by tenth` line
and `adapter`.

In [ ]:
%cd /content/nib
!python scripts/adapt_emuru.py --steps 2000 --device cuda

!mkdir -p /content/drive/MyDrive/nib/checkpoints
!cp /content/nib/checkpoints/emuru_cvl_r8_s2000.pt /content/nib/checkpoints/emuru_cvl_r8_s2000.json /content/drive/MyDrive/nib/checkpoints/
!ls -lh /content/drive/MyDrive/nib/checkpoints/

## 7o. Does it copy a new hand better now — 150 lines, about 35 minutes

The same first 150 requests as 7c, the same quality control, the only change
being the adapter from 7n. The writers here were not trained on.

**Compare with 7c's first 150 lines: identity 64.6%**, measured line for line
against one reference. The criterion, set before the run: beat it clearly.

**Send back:** the `adapter` line, the `HWD` block with `identity`, the `SUMMARY`
and the `selection` lines.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator emuru --samples 150 --style-refs 4 --candidates 4   --adapter checkpoints/emuru_cvl_r8_s2000.pt --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs4_cand4_emuru_cvl_r8_s2000 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs4_cand4_emuru_cvl_r8_s2000/generated | wc -l) generated images"

## 8. Eruku, for the comparison — optional, 128 minutes last time

Eruku runs at 35 seconds a line: classifier-free guidance is two forward passes
per token rather than one, which is what buys its text fidelity and what costs
the time.

Worth doing because the Emuru-against-Eruku comparison made on 2026-09-10 was
decided by a metric that measures sharpness, and Eruku's output may simply be
softer. HWD's distance and identity blocks will say. The cell saves itself to
Drive.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator eruku \
    --samples 300 \
    --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_eruku_lines /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_eruku_lines/generated | wc -l) generated images"

## 9. Compare what ran — no GPU, seconds

Reads what each run saved and reports whether the intervals separate at all.
Two figures whose intervals overlap are not a difference.

In [ ]:
%cd /content/nib
!python scripts/analyse_run.py outputs/eval_emuru_lines outputs/eval_eruku_lines

---

# Kept for reference — do not run without a reason

**Re-measuring the references** (`check_metrics.py`). They are measured and
committed in `references/`. CER cannot be computed here anyway, and the cell
downloads 1.4 GB of TrOCR in order to skip it.

**The guidance sweep.** Closed negative on 2026-09-10: cfg 1.0 / 1.25 / 2.0 gave
retrieval 10.0% / 5.3% / 6.7% with every interval overlapping, and cfg 2.0 was
clearly worse (15% truncated). Even cfg 1.0's upper bound sat below Emuru. It
would be worth revisiting only once HWD has replaced the metric that judged it.

**More than one style line.** Two are measured in cell 7b. `--style-refs 4`
breaks Emuru (0.25x of real width) because the prefix outgrows what its stopping
heuristic tolerates; the way to more lines is fixing that rule, not this flag.

In [ ]:
# %cd /content/nib
# !python scripts/check_metrics.py --pack data/processed/cvl_lines_64.lmdb --samples 300 --device cuda
# !python scripts/evaluate_generator.py --generator eruku-no-style-text --samples 300 --device cuda

## What to report back

- the `SUMMARY` block from each run, **with its intervals**
- the whole `HWD` block — the distance lines, the percentage, and the
  `identity` block with its last line. This is the one that matters
- `truncated` and `empty outputs`
- the `saved to Drive` line
- anything that failed, with the full error text